# Wound Care RAG — Vector Store Ingestion (v3)

Reads 4 curated chunk JSON files from `ingestion_output_ai/`, embeds each chunk's  
`ai_summary` as `page_content`, stores raw `text` + full provenance as metadata,  
and persists to ChromaDB at `db_wound_care_v3/`.

**Embedding model**: `abhinand/MedEmbed-large-v0.1`  
**Similarity metric**: cosine (HNSW)

---
### Chunk JSON schema (all 4 files share this)
```
chunk_id       — unique hex ID
source         — original PDF filename
section        — section heading
parent_section — parent section heading
chunk_index    — positional index within source
char_count     — character count of raw text
text           — raw extracted text  → stored as metadata
ai_summary     — AI-cleaned summary  → used as page_content for embedding
```

## 0 · Install dependencies

In [1]:
# Run once; restart kernel after if installing for the first time
# !pip install langchain langchain-community chromadb sentence-transformers torch

## 1 · Imports & config

In [4]:
import json
import os
import shutil
from pathlib import Path
from typing import List

import torch
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [5]:
# ── Paths ─────────────────────────────────────────────────────────────────────
CHUNKS_DIR = "./ingestion_output_ai"
DB_DIR     = "./db_wound_care_v3"

# ── Chunk JSON files  (filename → path) ──────────────────────────────────────
CHUNK_JSON_FILES = {
    "AJGP" : os.path.join(CHUNKS_DIR, "AJGP_wound_dressings_kept.json"),
    "GP"   : os.path.join(CHUNKS_DIR, "GP_wound_dressings_kept.json"),
    "SFP"  : os.path.join(CHUNKS_DIR, "SFP_wound_dressings_kept.json"),
    "WCM"  : os.path.join(CHUNKS_DIR, "WCM_wound_care_manual_kept.json"),
}

# ── Guideline metadata keyed by source PDF filename ──────────────────────────
# Used to enrich every chunk's metadata with authority / year / focus
GUIDELINE_METADATA = {
    "The art and science of selecting appropriate dressings for acute open wounds in general practice.pdf": {
        "guideline_type" : "clinical_review",
        "authority"      : "RACGP_Australia",
        "year"           : "2022",
        "focus"          : "acute_wound_dressings_general_practice",
    },
    "Garis Panduan Perkhidmatan Penjagaan Luka di Fasiliti Kesihatan Primer.pdf": {
        "guideline_type" : "national_guideline",
        "authority"      : "MOH_Malaysia",
        "year"           : "2019",
        "focus"          : "primary_care_wound_service",
    },
    "Wound Dressings - A Primer For The Family Physician.pdf": {
        "guideline_type" : "clinical_education",
        "authority"      : "Singapore_Family_Physician",
        "year"           : "2018",
        "focus"          : "wound_management_primary_care",
    },
    "Wound Care Manual - First Edition.pdf": {
        "guideline_type" : "clinical_manual",
        "authority"      : "MOH_Malaysia",
        "year"           : "2014",
        "focus"          : "comprehensive_wound_care",
    },
}

# ── Verify paths ──────────────────────────────────────────────────────────────
print("📂 Path check:")
print(f"   CHUNKS_DIR : {CHUNKS_DIR}  {'✅' if os.path.isdir(CHUNKS_DIR) else '❌ NOT FOUND'}")
print(f"   DB_DIR     : {DB_DIR}  {'✅ exists (will overwrite)' if os.path.isdir(DB_DIR) else '(will be created)'}")

print("\n📋 Chunk JSON files:")
all_ok = True
for key, path in CHUNK_JSON_FILES.items():
    exists = os.path.isfile(path)
    print(f"   {'✅' if exists else '❌'} [{key}] {path}")
    if not exists:
        all_ok = False

if not all_ok:
    print("\n⚠️  One or more JSON files missing — update CHUNK_JSON_FILES paths above.")
else:
    print("\n✅ All JSON files found.")

📂 Path check:
   CHUNKS_DIR : ./ingestion_output_ai  ✅
   DB_DIR     : ./db_wound_care_v3  (will be created)

📋 Chunk JSON files:
   ✅ [AJGP] ./ingestion_output_ai\AJGP_wound_dressings_kept.json
   ✅ [GP] ./ingestion_output_ai\GP_wound_dressings_kept.json
   ✅ [SFP] ./ingestion_output_ai\SFP_wound_dressings_kept.json
   ✅ [WCM] ./ingestion_output_ai\WCM_wound_care_manual_kept.json

✅ All JSON files found.


## 2 · Load chunks from JSON files

In [6]:
def load_chunks(json_paths: dict) -> List[dict]:
    """
    Load kept_chunks from all JSON files.
    Supports two formats:
      - dict with 'kept_chunks' key  (AJGP / GP / SFP)
      - bare list at root level      (WCM)
    Returns a flat list of chunk dicts.
    """
    all_chunks = []

    for label, path in json_paths.items():
        if not os.path.isfile(path):
            print(f"   ⚠️  Skipping missing file: {path}")
            continue

        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Handle both formats
        if isinstance(data, list):
            chunks = data
            # No meta block — derive stats directly from the list
            total_chunks  = len(chunks)
            ai_summarised = sum(1 for c in chunks if c.get("ai_summary", "").strip())
        else:
            chunks        = data.get("kept_chunks", [])
            meta          = data.get("meta", {})
            total_chunks  = meta.get("total_chunks",  len(chunks))
            ai_summarised = meta.get("ai_summarised", "?")

        print(f"   📄 [{label}] {os.path.basename(path)}: "
              f"{len(chunks)} chunks "
              f"(total={total_chunks}, "
              f"ai_summarised={ai_summarised})")

        all_chunks.extend(chunks)

    print(f"\n   ✅ Total chunks loaded: {len(all_chunks)}")
    return all_chunks


all_chunks = load_chunks(CHUNK_JSON_FILES)

   📄 [AJGP] AJGP_wound_dressings_kept.json: 19 chunks (total=19, ai_summarised=19)
   📄 [GP] GP_wound_dressings_kept.json: 13 chunks (total=13, ai_summarised=13)
   📄 [SFP] SFP_wound_dressings_kept.json: 36 chunks (total=36, ai_summarised=36)
   📄 [WCM] WCM_wound_care_manual_kept.json: 40 chunks (total=40, ai_summarised=40)

   ✅ Total chunks loaded: 108


## 3 · Convert chunks → LangChain Documents

- **`page_content`** = `ai_summary` (what gets embedded)
- **`metadata`** = everything else, including raw `text` for evidence retrieval

In [7]:
def chunks_to_documents(
    chunks: List[dict],
    guideline_metadata: dict,
) -> List[Document]:
    """
    Convert raw chunk dicts to LangChain Documents.

    page_content → ai_summary  (embedded into ChromaDB)
    metadata     → full provenance for evidence display & filtering:
        chunk_id, source, section, parent_section, chunk_index,
        char_count, raw_text,
        guideline_type, authority, year, focus  (from GUIDELINE_METADATA lookup)
    """
    documents = []
    skipped   = 0

    for chunk in chunks:
        ai_summary = chunk.get("ai_summary", "").strip()
        raw_text   = chunk.get("text",       "").strip()
        source     = chunk.get("source",     "unknown")

        # Use ai_summary as page_content; fall back to raw_text if missing
        page_content = ai_summary if ai_summary else raw_text

        if not page_content:
            skipped += 1
            print(f"   ⚠️  Empty content, skipping chunk_id={chunk.get('chunk_id')}")
            continue

        # Look up guideline-level metadata by PDF filename
        guide_meta = guideline_metadata.get(source, {})

        metadata = {
            # ── Chunk identity ────────────────────────────────────────────
            "chunk_id"       : chunk.get("chunk_id",      ""),
            "chunk_index"    : chunk.get("chunk_index",    0),
            # ── Source provenance ─────────────────────────────────────────
            "source"         : source,
            "section"        : chunk.get("section",       ""),
            "parent_section" : chunk.get("parent_section",""),
            "char_count"     : chunk.get("char_count",     0),
            # ── Raw text (for evidence display, not embedded) ─────────────
            "raw_text"       : raw_text,
            # ── Guideline-level metadata (for filtering / citation) ────────
            "guideline_type" : guide_meta.get("guideline_type", "unknown"),
            "authority"      : guide_meta.get("authority",      "unknown"),
            "year"           : guide_meta.get("year",           "unknown"),
            "focus"          : guide_meta.get("focus",          "unknown"),
        }

        documents.append(Document(page_content=page_content, metadata=metadata))

    print(f"\n✅ Documents created : {len(documents)}")
    print(f"   Skipped (empty)   : {skipped}")
    return documents


documents = chunks_to_documents(all_chunks, GUIDELINE_METADATA)

# Quick sanity check — print first document
print("\n── Sample document ─────────────────────────────────────")
d = documents[0]
print(f"page_content (first 300 chars):\n{d.page_content[:300]}")
print(f"\nmetadata:")
for k, v in d.metadata.items():
    val_preview = str(v)[:120] if k != "raw_text" else str(v)[:80] + "..."
    print(f"   {k:<16}: {val_preview}")


✅ Documents created : 108
   Skipped (empty)   : 0

── Sample document ─────────────────────────────────────
page_content (first 300 chars):
Acute open wounds are a significant concern in general practice, necessitating careful selection of wound dressings. This summary provides a practical guide for choosing appropriate dressings for treating acute open wounds.

Key considerations for dressing selection include:

1. **Wound Characterist

metadata:
   chunk_id        : 65414974ff6f
   chunk_index     : 0
   source          : The art and science of selecting appropriate dressings for acute open wounds in general practice.pdf
   section         : Background & Objective
   parent_section  : Article Context
   char_count      : 780
   raw_text        : Background
Acute open wounds constitute a 
significant part of general practice....
   guideline_type  : clinical_review
   authority       : RACGP_Australia
   year            : 2022
   focus           : acute_wound_dressings_general_pract

## 4 · Build ChromaDB vector store

In [8]:
def create_vector_store(
    documents: List[Document],
    persist_directory: str,
    overwrite: bool = True,
):
    """
    Embed documents with MedEmbed-large-v0.1 and persist to ChromaDB.
    Uses cosine similarity (HNSW index).
    """
    if overwrite and os.path.isdir(persist_directory):
        print(f"🗑️  Removing existing DB at {persist_directory}...")
        shutil.rmtree(persist_directory)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🔮 Loading MedEmbed-large-v0.1 on {device}...")

    embedding_model = HuggingFaceEmbeddings(
        model_name="abhinand/MedEmbed-large-v0.1",
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True},
    )

    print(f"📦 Indexing {len(documents)} documents → {persist_directory} ...")

    # Batch in groups of 50 to avoid memory spikes on CPU
    BATCH_SIZE = 50
    vectorstore = None

    for i in range(0, len(documents), BATCH_SIZE):
        batch = documents[i : i + BATCH_SIZE]
        print(f"   [{i+1:>3}–{min(i+BATCH_SIZE, len(documents)):>3}] embedding {len(batch)} docs...")

        if vectorstore is None:
            # First batch — create the store
            vectorstore = Chroma.from_documents(
                documents          = batch,
                embedding          = embedding_model,
                persist_directory  = persist_directory,
                collection_metadata= {"hnsw:space": "cosine"},
            )
        else:
            # Subsequent batches — add to existing store
            vectorstore.add_documents(batch)

    print(f"\n✅ Vector store saved → {persist_directory}")
    print(f"   Total documents indexed: {vectorstore._collection.count()}")
    return vectorstore


vectorstore = create_vector_store(documents, DB_DIR, overwrite=True)

🔮 Loading MedEmbed-large-v0.1 on cuda...


C:\Users\GIGA\AppData\Local\Temp\ipykernel_16832\3653260060.py:17: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3448.74it/s]


📦 Indexing 108 documents → ./db_wound_care_v3 ...
   [  1– 50] embedding 50 docs...
   [ 51–100] embedding 50 docs...
   [101–108] embedding 8 docs...

✅ Vector store saved → ./db_wound_care_v3
   Total documents indexed: 108


## 5 · Smoke test — verify retrieval works

In [9]:
# ── Test query: mimics a typical TIME-input query ─────────────────────────────
TEST_QUERY = (
    "Wound with 50% slough and 50% granulation tissue, moderate exudate, "
    "non-advancing edges, no signs of infection. What dressing should I use?"
)

results = vectorstore.similarity_search_with_score(TEST_QUERY, k=5)

print(f"🔍 Query: {TEST_QUERY}\n")
print("── Top 5 results ────────────────────────────────────────────────────")
for rank, (doc, score) in enumerate(results, start=1):
    m = doc.metadata
    print(f"\n#{rank}  score={score:.4f}")
    print(f"   source    : {m['source']}")
    print(f"   section   : {m['section']}")
    print(f"   authority : {m['authority']} ({m['year']})")
    print(f"   chunk_id  : {m['chunk_id']}")
    print(f"   content   : {doc.page_content[:250]}...")

🔍 Query: Wound with 50% slough and 50% granulation tissue, moderate exudate, non-advancing edges, no signs of infection. What dressing should I use?

── Top 5 results ────────────────────────────────────────────────────

#1  score=0.2425
   source    : Garis Panduan Perkhidmatan Penjagaan Luka di Fasiliti Kesihatan Primer.pdf
   section   : Treatment Recommendation — Wound Type 5
   authority : MOH_Malaysia (2019)
   chunk_id  : aad7a40107b0
   content   : **Clinical Summary for Wound Type 5**

**Wound Characteristics:**
- Type: Dry, non-infected wound
- Necrosis/Slough: Greater than 25% of wound surface
- Infection: No wound infection present
- Moisture/Exudate: Dry to minimal
- Hospital Referral: Not...

#2  score=0.2518
   source    : Garis Panduan Perkhidmatan Penjagaan Luka di Fasiliti Kesihatan Primer.pdf
   section   : Wound Assessment — Decision Algorithm
   authority : MOH_Malaysia (2019)
   chunk_id  : bd2bb8e1321e
   content   : **Wound Assessment Algorithm Summary**

The wo

## 6 · (Optional) Reload DB and verify persistence

In [ ]:
# Reload from disk — use this in your RAG retrieval pipeline
device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = HuggingFaceEmbeddings(
    model_name="abhinand/MedEmbed-large-v0.1",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

loaded_db = Chroma(
    persist_directory  = DB_DIR,
    embedding_function = embedding_model,
    collection_metadata= {"hnsw:space": "cosine"},
)

count = loaded_db._collection.count()
print(f"✅ DB reloaded from {DB_DIR}")
print(f"   Documents in collection: {count}")

# Peek at metadata fields available for filtering
sample = loaded_db.get(limit=1, include=["metadatas"])
print(f"\n   Metadata fields: {list(sample['metadatas'][0].keys())}")